In [0]:
select * from `workspace`.`default`.`bright_tv_viewership` limit 100;

select* from `workspace`.`default`.`bright_tv_viewers_profile` limit 100;

-- CONVERSION: UTC to SAST
SELECT
    UserID0,
    Channel2,
    RecordDate2                                            AS RecordDate_UTC,
    RecordDate2 + INTERVAL 2 HOURS                         AS RecordDate_SAST,
    DATE(RecordDate2 + INTERVAL 2 HOURS)                   AS sa_date,
    HOUR(RecordDate2 + INTERVAL 2 HOURS)                   AS sa_hour,
    `Duration 2`
FROM `workspace`.`default`.`bright_tv_viewership`;


---Joining the tables
SELECT *
FROM `workspace`.`default`.`bright_tv_viewership`  v
JOIN `workspace`.`default`.`bright_tv_viewers_profile` p
ON v.UserID0 = p.UserID;


-- Sessions per day
SELECT
    DATE(RecordDate2 + INTERVAL 2 HOURS)            AS sa_date,
    COUNT(*)                                        AS sessions
FROM `workspace`.`default`.`bright_tv_viewership`
GROUP BY sa_date
ORDER BY sa_date;


-- Sessions per hour
SELECT
    HOUR(RecordDate2 + INTERVAL 2 HOURS)            AS sa_hour,
    COUNT(*)                                        AS sessions
FROM `workspace`.`default`.`bright_tv_viewership`
GROUP BY sa_hour
ORDER BY sa_hour;


-- Most active province
SELECT
    p.Province,
    COUNT(*)                                        AS sessions
FROM `workspace`.`default`.`bright_tv_viewership` v
JOIN `workspace`.`default`.`bright_tv_viewers_profile` p
    ON v.UserID0 = p.UserID
WHERE p.Province IS NOT NULL
  AND TRIM(p.Province) != ''
GROUP BY p.Province
ORDER BY sessions DESC;


-- Most active age groups
SELECT
    CASE
        WHEN p.Age < 18              THEN 'Under 18'
        WHEN p.Age BETWEEN 18 AND 24 THEN '18-24'
        WHEN p.Age BETWEEN 25 AND 34 THEN '25-34'
        WHEN p.Age BETWEEN 35 AND 44 THEN '35-44'
        WHEN p.Age BETWEEN 45 AND 54 THEN '45-54'
        WHEN p.Age >= 55             THEN '55+'
    END                                             AS age_band,
    COUNT(*)                                        AS sessions
FROM `workspace`.`default`.`bright_tv_viewership` v
JOIN `workspace`.`default`.`bright_tv_viewers_profile` p
    ON v.UserID0 = p.UserID
WHERE p.Age BETWEEN 5 AND 100
GROUP BY age_band
ORDER BY sessions DESC;


-- Identify the 15 lowest consumption days
SELECT
    DATE(RecordDate2 + INTERVAL 2 HOURS)                           AS sa_date,
    DAYOFWEEK(RecordDate2 + INTERVAL 2 HOURS)                      AS dow_number,
    COUNT(*)                                                        AS sessions,
    COUNT(DISTINCT UserID0)                                          AS unique_viewers
FROM `workspace`.`default`.`bright_tv_viewership`
GROUP BY sa_date, dow_number
ORDER BY sessions ASC
LIMIT 15;


-- Count NULLs and blanks across all profile columns
SELECT
    COUNT(*)                                                        AS total_profiles,
    SUM(CASE WHEN Gender   IS NULL OR TRIM(Gender)   = '' THEN 1 ELSE 0 END) AS gender_null,
    SUM(CASE WHEN Race     IS NULL OR TRIM(Race)     = '' THEN 1 ELSE 0 END) AS race_null,
    SUM(CASE WHEN Province IS NULL OR TRIM(Province) = '' THEN 1 ELSE 0 END) AS province_null,
    SUM(CASE WHEN Age      IS NULL OR Age NOT BETWEEN 5 AND 100    THEN 1 ELSE 0 END) AS age_suspect
FROM `workspace`.`default`.`bright_tv_viewers_profile`;

-- Count NULLs across viewership columns
SELECT
    COUNT(*)                                                        AS total_sessions,
    SUM(CASE WHEN UserID0       IS NULL THEN 1 ELSE 0 END)       AS userid_null,
    SUM(CASE WHEN Channel2        IS NULL
             OR TRIM(Channel2)   = ''    THEN 1 ELSE 0 END)        AS channel_null,
    SUM(CASE WHEN RecordDate2     IS NULL THEN 1 ELSE 0 END)       AS date_null,
    SUM(CASE WHEN `Duration 2` IS NULL THEN 1 ELSE 0 END)        AS duration_null
FROM `workspace`.`default`.`bright_tv_viewership`;


-- Show all profiles where Gender is NULL or blank
SELECT *
FROM `workspace`.`default`.`bright_tv_viewers_profile`
WHERE Gender IS NULL
   OR TRIM(Gender) = '';


   -- Race analysis NOT NULL
SELECT
    p.Race,
    COUNT(*)                                                        AS sessions
FROM `workspace`.`default`.`bright_tv_viewership` v
JOIN `workspace`.`default`.`bright_tv_viewers_profile` p
  ON v.UserID0  = p.UserID
WHERE p.Race IS NOT NULL
  AND TRIM(p.Race) != ''
GROUP BY p.Race
ORDER BY sessions DESC;